### Задание 1
Используя `RecursiveCharacterTextSplitter`, разбейте text на фрагменты размером не более 150 символов с перекрытием в 30 символов. В качестве разделителя используйте набор разделителей ["\n\n", "\n", ". ", " "]. 

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text = """Глубокое обучение в медицинской диагностике.
Введение. Современные CNN достигают высокой точности при анализе рентгеновских снимков.
Методы. Мы сравнивали ResNet-50 и Vision Transformer на наборе данных CheXpert.
Результаты. ViT показал преимущество для выявления пневмонии.
Обсуждение. Несмотря на прогресс, сохраняются проблемы:
1) Нехватка размеченных данных.
2) "Чёрный ящик" принятия решений.
Заключение. Перспективным направлением является ..."""


splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " "], # разбиваем текст по символу переноса строки
)

chunks = splitter.split_text(text)
for i, chunk in enumerate(chunks):
    print(f"chunk {i+1}: {chunk}\n---")

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 7/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


chunk 1: Глубокое обучение в медицинской диагностике.
Введение. Современные CNN достигают высокой точности при анализе рентгеновских снимков.
---
chunk 2: Методы. Мы сравнивали ResNet-50 и Vision Transformer на наборе данных CheXpert.
Результаты. ViT показал преимущество для выявления пневмонии.
---
chunk 3: Обсуждение. Несмотря на прогресс, сохраняются проблемы:
1) Нехватка размеченных данных.
2) "Чёрный ящик" принятия решений.
---
chunk 4: Заключение. Перспективным направлением является ...
---


### Задание 2
Проинициализируйте индекс и добавьте в векторную БД тексты с метаданными. Осуществите поиск по всем документам, кроме категории "fantasy" с запросом `"Зачем нужен RAG ?"`. Верните один наиболее релевантный документ.

In [10]:
import faiss

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.documents import Document


chunk1 = "RAG позволяет расширить знания LLM"
chunk2 = "кросс-энкодер позволят проводить ранжирование отобранных результатов"
chunk3 = "RAG = robust attack on giants"

chunks = [chunk1, chunk2, chunk3]
metadata = ["RAG", "ML", "fantasy"]

docs = [
    Document(
        page_content=c,
        metadata={"source": m},
    ) for c, m in zip(chunks, metadata)
]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))


vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)


vector_store.add_documents(docs)
vector_store.similarity_search("Зачем нужен RAG ?", k=1, filter={"source": {"$neq": "fantasy"}}) 


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5775.50it/s]


[Document(id='7b9201b8-7cce-4602-985e-287d36422577', metadata={'source': 'RAG'}, page_content='RAG позволяет расширить знания LLM')]